In [ ]:
import os
import time
import json
import requests
import pandas as pd
from glob import glob
from tqdm import tqdm

In [ ]:
root_path = r'C:\Users\fienn\Documents\project\da_team_project_4\data\github_bckup'
matchid_path = os.path.join(root_path, '0_data', '16_matchids')
output_path = os.path.join(root_path, '0_data', '17_match_details')
os.makedirs(output_path, exist_ok=True)

In [ ]:
API_KEY=""
HEADERS = {"X-Riot-Token": API_KEY}

REGION_MAPPING = {
    'KR': 'asia', 'JP1': 'asia',
    'BR1': 'americas', 'NA1': 'americas', 'LA1': 'americas', 'LA2': 'americas',
    'EUN1': 'europe', 'EUW1': 'europe', 'TR1': 'europe', 'RU': 'europe', 'ME1': 'europe',
    'OC1': 'sea', 'SG2': 'sea', 'TW2': 'sea', 'VN2': 'sea'
}

def get_match_detail(match_id, retries=3):
    """매치 ID에 대한 request"""
    platform = match_id.split('_')[0]
    routing = REGION_MAPPING.get(platform.upper(), '')
    url = f"https://{routing}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS)
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 429:
                wait_time = int(response.headers.get("Retry-After", 10))
                print(f"⚠️ 429 Limit. {wait_time}초 대기...")
                time.sleep(wait_time + 1)
                continue
            elif response.status_code == 404:
                print(f"Match not found: {match_id}")
                return None
            elif response.status_code == 502:
                print(f"502 Error. 리트 {attempt+1}/{retries}): {match_id}. 3초 후 재시도...")
                time.sleep(3)
                continue
            else:
                print(f"Error {response.status_code} for {match_id}")
                return None
        except Exception as e:
            print(f"Connection Failed: {e}")
            time.sleep(0.8)
    print(f"{retries}번 재시도 실패: {match_id} 건너뜀.")
    return None

def run_detail_collection(batch_size=1000, max_lines_per_file=100000):
    # 모든 매치 ID 파일 로드 (sampled_collected_match_ids_*.csv)
    '''매치id 파일'''
    id_files = glob(os.path.join(matchid_path, "sampled_collected_match_ids_*.csv"))

    # test file
    '''2000개짜리 테스트용 파임
    실제 수집할때는 바꿔야함'''
    # id_files = glob(os.path.join(matchid_path, "sampled_collected_match_ids_fienn.csv"))
    
    all_match_ids = []
    for f in id_files:
        df = pd.read_csv(f)
        all_match_ids.extend(df['match_id'].tolist())
    
    print(f"📊 총 수집 대상 매치 ID: {len(all_match_ids)}개")

    # 이미 수집된 매치 ID 확인 (이어받기용)
    # 각 지역별로 결과 파일이 생기므로, 출력 폴더 내 모든 jsonl을 확인.
    processed_ids = set()
    for f in glob(os.path.join(output_path, "*.jsonl")):
        with open(f, 'r', encoding='utf-8', errors='replace') as reader:
            for line in reader:
                try:
                    data = json.loads(line)
                    processed_ids.add(data['metadata']['matchId'])
                except: continue
    
    print(f"✅ 이미 완료된 매치: {len(processed_ids)}개")
    remaining_ids = [m for m in all_match_ids if m not in processed_ids]
    print(f"🚀 새로 수집할 매치: {len(remaining_ids)}개")

    '''여기부터 수집 로직'''
    buffer = []
    total_new = len(remaining_ids)
    
    print(f"🚀 수집 시작 (배치 사이즈: {batch_size})")
    
    # 지역별 파일 파트 번호 관리 (예: details_kr_part1.jsonl)
    # file_parts = {} 

    # for idx, m_id in enumerate(tqdm(remaining_ids, desc="수집 중")):
    #     platform = m_id.split('_')[0].lower()
    #     detail = get_match_detail(m_id)
        
    #     if detail:
    #         buffer.append((platform, detail))
        
    #     # 배치 저장 시점
    #     if len(buffer) >= batch_size or (idx + 1 == total_new):
    #         for plat in set(p for p, d in buffer):
    #             plat_data = [d for p, d in buffer if p == plat]
                
    #             # 파일 분할 로직
    #             part_num = file_parts.get(plat, 1)
    #             output_file = os.path.join(output_path, f"details_{plat}_part{part_num}.jsonl")
                
    #             # 현재 파일의 라인 수 확인 (파일이 크면 다음 파트로)
    #             if os.path.exists(output_file):
    #                 with open(output_file, 'r', encoding='utf-8') as f:
    #                     line_count = sum(1 for _ in f)
    #                 if line_count >= max_lines_per_file:
    #                     part_num += 1
    #                     file_parts[plat] = part_num
    #                     output_file = os.path.join(output_path, f"details_{plat}_part{part_num}.jsonl")

    #             with open(output_file, 'a', encoding='utf-8') as f:
    #                 for data in plat_data:
    #                     f.write(json.dumps(data, ensure_ascii=False) + '\n')
            
    #         buffer.clear()
# ===============================================================================
# 260217 edited
# ===============================================================================
    file_status = {} 

    for idx, m_id in enumerate(tqdm(remaining_ids, desc="수집 중")):
        platform = m_id.split('_')[0].lower()
        detail = get_match_detail(m_id)
        
        if detail:
            buffer.append((platform, detail))
        
        if len(buffer) >= batch_size or (idx + 1 == total_new):
            for plat in set(p for p, d in buffer):
                plat_data = [d for p, d in buffer if p == plat]
                
                # 처음 보는 플랫폼이면 초기화
                if plat not in file_status:
                    # 폴더 내 기존 파일 확인해서 마지막 파트와 줄 수 파악 (최초 1회만)
                    existing_files = sorted(glob(os.path.join(output_path, f"details_{plat}_part*.jsonl")))
                    if existing_files:
                        last_file = existing_files[-1]
                        part_num = int(last_file.split('_part')[-1].split('.jsonl')[0])
                        with open(last_file, 'rb') as f:
                            line_count = sum(1 for _ in f)
                        file_status[plat] = [part_num, line_count]
                    else:
                        file_status[plat] = [1, 0]
    
                part_num, current_lines = file_status[plat]
                
                # 저장 전 파일 분할 체크
                if current_lines + len(plat_data) > max_lines_per_file:
                    part_num += 1
                    current_lines = 0
                    file_status[plat] = [part_num, current_lines]
    
                output_file = os.path.join(output_path, f"details_{plat}_part{part_num}.jsonl")
                
                with open(output_file, 'a', encoding='utf-8') as f:
                    for data in plat_data:
                        f.write(json.dumps(data, ensure_ascii=False) + '\n')
                
                # 메모리 내 줄 수 업데이트 (파일 안 읽음!)
                file_status[plat][1] += len(plat_data)
                
            buffer.clear()
    
            # 진행상황 로그
            if (idx + 1) % 5000 == 0:
                print(f"🚀 [진행도] {idx+1}/{total_new} ({((idx+1)/total_new)*100:.2f}%) 완료")

        time.sleep(0.5)

In [ ]:
id_files = glob(os.path.join(matchid_path, "sampled_collected_match_ids_*.csv"))
id_files

In [ ]:
run_detail_collection(batch_size=1000)

In [ ]:
run_detail_collection(batch_size=1000)

In [ ]:
run_detail_collection(batch_size=1000)

## 확인

In [ ]:
detail_path = r'C:\Users\fienn\Documents\project\da_team_project_4\data\github_bckup\0_data\17_match_details'
jsonl_files = glob(os.path.join(detail_path, "*.jsonl"))

total_matches = 0
print("지역별 수집 현황:")
print("-" * 30)

for file in jsonl_files:
    file_name = os.path.basename(file)
    with open(file, 'r', encoding='utf-8') as f:
        # 파일의 줄 수를 세는 가장 효율적인 방법
        count = sum(1 for line in f)
        print(f"{file_name:30} : {count:,} 개")
        total_matches += count

print("-" * 30)
print(f"✅ 총 수집된 매치 상세 정보: {total_matches:,} 개")

In [ ]:
# 가장 최근에 수정된 파일 하나 확인
if jsonl_files:
    sample_file = jsonl_files[0]
    with open(sample_file, 'r', encoding='utf-8') as f:
        first_line = f.readline()
        data = json.loads(first_line)
        
        # 주요 정보 출력 테스트
        match_id = data['metadata']['matchId']
        game_version = data['info']['gameVersion']
        print(f"\n🔍 샘플 데이터 확인 ({os.path.basename(sample_file)}):")
        print(f"   - 매치 ID: {match_id}")
        print(f"   - 게임 버전: {game_version}")
        print(f"   - 참가자 수: {len(data['info']['participants'])}명")

## Timeline api

In [ ]:
root_path = r'C:\Users\fienn\Documents\project\da_team_project_4\data\github_bckup'
detail_path = os.path.join(root_path, '0_data', '17_match_details')
timeline_output_path = os.path.join(root_path, '0_data', '18_match_timelines')
os.makedirs(timeline_output_path, exist_ok=True)

In [ ]:
def get_match_timeline(match_id, retries=3):
    """매치 ID에 대한 타임라인 정보 요청"""
    platform = match_id.split('_')[0]
    routing = REGION_MAPPING.get(platform.upper())
    # 엔드포인트
    url = f"https://{routing}.api.riotgames.com/lol/match/v5/matches/{match_id}/timeline"
    
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS)
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 429:
                wait_time = int(response.headers.get("Retry-After", 10))
                time.sleep(wait_time + 1)
                continue
            elif response.status_code == 502:
                time.sleep(3)
                continue
            elif response.status_code == 404:
                return None
            else:
                return None
        except Exception as e:
            time.sleep(5)
    return None

def run_timeline_collection_from_details(batch_size=200, max_lines_per_file=50000):
    """
    수집 완료된 Match Details 파일을 읽어 타임라인 수집 대상을 선정함
    """
    # Match Details 파일들로부터 매치 ID 추출
    detail_files = glob(os.path.join(detail_path, "*.jsonl"))
    target_match_ids = []
    EXCLUDE_PREFIXES = ('PH2_', 'TH2_')

    print("🔍 상세 정보 파일에서 매치 ID 추출 중...")
    for f_path in detail_files:
        with open(f_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    m_id = data['metadata']['matchId']
                    
                    # PH2, TH2 제외 필터링
                    if not m_id.startswith(EXCLUDE_PREFIXES):
                        target_match_ids.append(m_id)
                except Exception as e:
                    continue
    
    target_match_ids = list(set(target_match_ids)) # 중복 제거
    print(f"📊 수집 대상 매치 ID 추출 완료: {len(target_match_ids):,} 개")

    # 이미 수집된 타임라인 제외 (이어받기)
    processed_ids = set()
    timeline_files = glob(os.path.join(timeline_output_path, "*.jsonl"))
    for f_path in timeline_files:
        with open(f_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    processed_ids.add(data['metadata']['matchId'])
                except: continue
    
    remaining_ids = [m for m in target_match_ids if m not in processed_ids]
    print(f"✅ 이미 완료된 타임라인: {len(processed_ids):,} 개")
    print(f"🚀 새로 수집할 타임라인: {len(remaining_ids):,} 개")

    # 수집
    buffer = []
    file_parts = {}

    for idx, m_id in enumerate(tqdm(remaining_ids, desc='수집 중')):
        timeline = get_match_timeline(m_id) 
        
        if timeline:
            platform = m_id.split('_')[0].lower()
            buffer.append((platform, timeline))
        
        if len(buffer) >= batch_size or (idx + 1 == len(remaining_ids)):
            for plat in set(p for p, t in buffer):
                plat_data = [t for p, t in buffer if p == plat]
                part_num = file_parts.get(plat, 1)
                
                output_file = os.path.join(timeline_output_path, f"timeline_{plat}_part{part_num}.jsonl")
                
                # 파일 분할 로직 (기존 파일 줄 수 확인)
                if os.path.exists(output_file):
                    with open(output_file, 'r', encoding='utf-8') as f:
                        if sum(1 for _ in f) >= max_lines_per_file:
                            part_num += 1
                            file_parts[plat] = part_num
                            output_file = os.path.join(timeline_output_path, f"timeline_{plat}_part{part_num}.jsonl")

                with open(output_file, 'a', encoding='utf-8') as f:
                    for data in plat_data:
                        f.write(json.dumps(data, ensure_ascii=False) + '\n')
            
            buffer.clear()
            if (idx + 1) % 100 == 0:
                print(f"⌛ [{idx+1}/{len(remaining_ids)}] 수집 중... (진행률: {((idx+1)/len(remaining_ids))*100:.2f}%)")

        time.sleep(0.05)

In [ ]:
run_timeline_collection_from_details()